# Exploratory Data Analysis: UCI Bike Sharing Dataset

**Project:** AI-Based Bicycle Rental Demand Prediction Using Generalized Additive Models (GAM)  
**Dataset Source:** [UCI Machine Learning Repository — Bike Sharing Dataset](https://archive.ics.uci.edu/dataset/275/bike+sharing+dataset)

This notebook explores the real hourly bicycle rental dataset (`data/raw/hour.csv`, 17,379 records from 2011-01-01 to 2012-12-31) to understand temporal patterns, weather nonlinearities, and the target (`cnt`) distribution.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
df = pd.read_csv('../data/raw/hour.csv')
print('Shape:', df.shape)
print('Date Range:', df['dteday'].min(), 'to', df['dteday'].max())
df.head()

## 1. Data Integrity & Target Distribution (`cnt`)
- **Missing values:** 0
- **Duplicate rows:** 0
- **Target `cnt`:** Right-skewed non-negative count variable (mean = 189.46, median = 142.0, std = 181.39, min = 1, max = 977, skewness = 1.28).
- Because `cnt` is non-negative and right-skewed count data, a **PoissonGAM with a log link** is well suited.

In [ ]:
print('Missing values:', df.isnull().sum().sum())
print('Duplicates:', df.duplicated().sum())
print(df['cnt'].describe())

fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(df['cnt'], bins=45, kde=True, color='#2563eb', ax=ax)
ax.set_title('Distribution of Hourly Bicycle Rentals (cnt)')
plt.show()

## 2. Temporal Patterns: Hour of Day & Working Day
- **Working days** exhibit a sharp bimodal commute pattern peaking around **8 AM** and **5–6 PM**.
- **Weekends and holidays** exhibit a single broad midday leisure peak between **11 AM and 4 PM**.
- These strong nonlinear hourly curves motivate smooth spline terms `s(hr)` in the GAM.

In [ ]:
hourly = df.groupby(['hr', 'workingday'])['cnt'].mean().reset_index()
hourly['Day Type'] = hourly['workingday'].map({0: 'Weekend/Holiday', 1: 'Working Day'})
fig, ax = plt.subplots(figsize=(10, 4))
sns.lineplot(data=hourly, x='hr', y='cnt', hue='Day Type', marker='o', ax=ax)
ax.set_title('Average Hourly Demand: Working Day vs Weekend/Holiday')
ax.set_xticks(range(0, 24))
plt.show()

## 3. Weather & Seasonal Effects
- Clear weather (`weathersit = 1`) has the highest average demand; rain/snow (`weathersit = 3, 4`) sharply reduces rentals.
- Temperature (`temp`) shows a positive but saturating nonlinear relationship with demand, while high humidity (`hum`) suppresses rentals.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.barplot(data=df, x='weathersit', y='cnt', ax=axes[0], errorbar=None)
axes[0].set_title('Average Demand by Weather Situation')
sns.barplot(data=df, x='season', y='cnt', ax=axes[1], errorbar=None)
axes[1].set_title('Average Demand by Season (1:Spring, 2:Summer, 3:Fall, 4:Winter)')
plt.tight_layout()
plt.show()